<div style="padding:26px 28px;border-radius:18px;background:linear-gradient(135deg,#101827,#18334a,#235b64);color:#f8fafc;margin:8px 0 20px 0;">
  <div style="font-size:12px;letter-spacing:1.7px;text-transform:uppercase;color:#a7f3d0;">Kaggle · Playground Series S6E10</div>
  <div style="font-size:30px;font-weight:800;line-height:1.25;margin-top:10px;">Original 0.96217 Baseline Lab</div>
  <div style="font-size:15px;line-height:1.65;color:#dbeafe;margin-top:12px;">A clean and reproducible starting point for future rank-blending experiments, anchored to the original u07_META105_xgb05 recipe.</div>
</div>

## 1. Objective

This notebook reconstructs the original `u07_META105_xgb05` candidate, historically reported with Public LB **0.96217**.

The original recipe is:

\[
R_{\text{baseline}} =
1.05R_{\text{MetaStack}} - 0.05R_{\text{XGBoost}}
\]

Each prediction vector is converted to normalized ascending ranks using `scipy.stats.rankdata(v) / N`. The weighted ranks are then combined using the original blend formula.

This notebook is the permanent baseline for future experiments. It does not use the later E37/E38 reconstructed candidates, H-blend, Summit, or OOF-based model selection.

## 2. Credits & Acknowledgements

Thank you to the competition organizers and the authors who shared the prediction assets and original blending method.

- **Competition:** [Kaggle — Predicting Airline Satisfaction](https://www.kaggle.com/competitions/playground-series-s6e10)
- **Original 0.96217 recipe:** [Areeb — S6E10 | Do You Need Less? LB 0.96217](https://www.kaggle.com/code/areebbb/s6e10-do-you-need-less-lb-0-96217)
- **MetaStack prediction:** [goodpjw2008 — S6E10 Meta Stack on Public OOF | LB 0.96213](https://www.kaggle.com/code/goodpjw2008/s6e10-meta-stack-on-public-oof-lb-0-96213)
- **XGBoost test prediction:** [Chris Deotte — S6E10 TFM OOF and Test Predictions](https://www.kaggle.com/datasets/cdeotte/s6e10-tfm-oof-and-test-predictions)

The original blend notebook also acknowledges H-blend, Summit Composer, Twin Stacks, TabPFN, Ensembling, Are You Satisfied and other public contributions. Those assets are not loaded in this baseline-only notebook because the `u07_META105_xgb05` recipe does not use their predictions. If later experiments use them, their corresponding acknowledgements will be retained.

This notebook independently reconstructs the published recipe from the referenced prediction vectors. It does not claim authorship of those predictions.

## 3. Required Inputs

1. Official competition: `Predicting Airline Satisfaction`
2. Dataset: `S6E10 TFM OOF and Test Predictions` — Chris Deotte
3. Notebook output: `S6E10 Meta Stack on Public OOF | LB 0.96213` — goodpjw2008

## 4. Experimental Policy

- Preserve `u07_META105_xgb05.csv` as the frozen reference.
- Never overwrite the frozen baseline when creating future candidates.
- Add future experiments as separate, named candidates with explicit formulas and source records.
- Do not treat rank correlation or an unvalidated proxy metric as proof of a leaderboard improvement.
- The historical Public LB 0.96217 is a reference score, not a score recalculated by this notebook.


In [ ]:

# ============================================================
# PAS E40 — Original 0.96217 Baseline Lab
# Full Cell: robust input discovery + exact rank-blend recipe
#
# Required Inputs:
# 1. Competition: Predicting Airline Satisfaction
# 2. Dataset: S6E10 TFM OOF and Test Predictions
# 3. Notebook: S6E10 Meta Stack on Public OOF by goodpjw2008
#
# No model training, no OOF scoring, no Kaggle submission.
# ============================================================

from pathlib import Path
import os
import json
import hashlib
import zipfile
import shutil

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from IPython.display import display, FileLink


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

OUT_DIR = WORK_ROOT / "PAS_E40_Original_096217_Baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_NAME = "u07_META105_xgb05"
EXPECTED_N_TEST = 299844
HISTORICAL_PUBLIC_LB = 0.96217

print("=" * 70)
print("PAS E40 — Original 0.96217 Baseline")
print("=" * 70)
print("Input root :", INPUT_ROOT)
print("Output dir :", OUT_DIR)


# ------------------------------------------------------------
# 2. Utility functions
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def norm_path(path):
    return (
        str(path)
        .lower()
        .replace("_", "-")
        .replace(" ", "-")
        .replace("\\", "/")
    )


def require_one(paths, description):
    paths = sorted(set(Path(p) for p in paths))

    if len(paths) != 1:
        shown = "\n".join(str(p) for p in paths[:30])
        if not shown:
            shown = "(no matches)"

        raise FileNotFoundError(
            f"\nExpected exactly one {description}; "
            f"found {len(paths)}.\n"
            f"Matches:\n{shown}\n"
            "Please check the mounted Kaggle Inputs."
        )

    return paths[0]


# ------------------------------------------------------------
# 3. Find official competition test.csv
# ------------------------------------------------------------

print("\n[1/6] Locating official competition...")

COMP_DIR = (
    INPUT_ROOT
    / "competitions"
    / "playground-series-s6e10"
)

if not (
    (COMP_DIR / "train.csv").is_file()
    and (COMP_DIR / "test.csv").is_file()
):
    comp_hits = [
        p.parent
        for p in INPUT_ROOT.rglob("test.csv")
        if (
            p.parent.name == "playground-series-s6e10"
            and (p.parent / "train.csv").is_file()
        )
    ]

    COMP_DIR = require_one(
        comp_hits,
        "official playground-series-s6e10 directory"
    )

TEST_PATH = COMP_DIR / "test.csv"

test_df = pd.read_csv(TEST_PATH, usecols=["id"])
IDS = test_df["id"].to_numpy()
N = len(IDS)

if N != EXPECTED_N_TEST:
    raise ValueError(
        f"Official test row count mismatch: "
        f"expected {EXPECTED_N_TEST}, found {N}."
    )

if pd.Series(IDS).duplicated().any():
    raise ValueError("Duplicate IDs detected in official test.csv.")

print("Competition directory:", COMP_DIR)
print("Test rows:", f"{N:,}")


# ------------------------------------------------------------
# 4. Find MetaStack submission.csv
#
# Previous code required the old 0.96213 slug.
# This version matches the notebook identity without requiring
# its score suffix to remain unchanged.
# ------------------------------------------------------------

print("\n[2/6] Locating MetaStack prediction...")

notebook_root = INPUT_ROOT / "notebooks"

if not notebook_root.is_dir():
    raise FileNotFoundError(
        "Kaggle Notebook Inputs directory not found: "
        f"{notebook_root}"
    )

all_submission_files = sorted(
    p
    for p in notebook_root.rglob("submission.csv")
    if p.is_file()
)

# Highest-confidence match: author + notebook identity.
meta_hits = [
    p
    for p in all_submission_files
    if (
        "goodpjw2008" in norm_path(p)
        and "meta-stack" in norm_path(p)
        and "public-oof" in norm_path(p)
    )
]

# Fallback: mounted path may omit the author name or change
# the score suffix, for example 0.96213 -> 0.96216.
if not meta_hits:
    meta_hits = [
        p
        for p in all_submission_files
        if (
            "meta-stack" in norm_path(p)
            and "public-oof" in norm_path(p)
        )
    ]

# If the notebook slug is not reflected in the path, do not
# silently choose an unrelated submission.csv.
if len(meta_hits) != 1:
    print("\nMetaStack automatic matching failed.")
    print("Matching candidates:", len(meta_hits))

    for p in meta_hits:
        print("  MATCH:", p)

    print("\nAll mounted Notebook submission.csv files:")
    for p in all_submission_files[:100]:
        print("  ", p)

    raise FileNotFoundError(
        "\nCould not uniquely identify the intended MetaStack "
        "submission.csv. Inspect the paths printed above. "
        "No baseline candidate has been generated."
    )

META_PATH = meta_hits[0]

print("MetaStack source:", META_PATH)


# ------------------------------------------------------------
# 5. Find Chris Deotte's XGBoost test prediction
# ------------------------------------------------------------

print("\n[3/6] Locating XGBoost prediction...")

TFM_DIR = (
    INPUT_ROOT
    / "datasets"
    / "cdeotte"
    / "s6e10-tfm-oof-and-test-predictions"
)

XGB_PATH = TFM_DIR / "test_xgboost.npy"

if not XGB_PATH.is_file():
    xgb_hits = [
        p
        for p in INPUT_ROOT.rglob("test_xgboost.npy")
        if (
            "s6e10-tfm-oof-and-test-predictions"
            in norm_path(p)
        )
    ]

    XGB_PATH = require_one(
        xgb_hits,
        "Chris Deotte TFM test_xgboost.npy"
    )

print("XGBoost source:", XGB_PATH)


# ------------------------------------------------------------
# 6. Load and validate the two prediction vectors
# ------------------------------------------------------------

print("\n[4/6] Loading and validating predictions...")

meta_df = pd.read_csv(META_PATH)

if "id" not in meta_df.columns:
    raise ValueError(
        "MetaStack submission.csv has no id column. "
        f"Columns: {meta_df.columns.tolist()}"
    )

if meta_df["id"].duplicated().any():
    raise ValueError(
        "Duplicate IDs detected in MetaStack submission.csv."
    )

if "satisfaction" in meta_df.columns:
    META_COL = "satisfaction"
else:
    numeric_cols = [
        c
        for c in meta_df.columns
        if (
            c != "id"
            and pd.api.types.is_numeric_dtype(meta_df[c])
        )
    ]

    if len(numeric_cols) != 1:
        raise ValueError(
            "Cannot uniquely identify the MetaStack prediction "
            f"column. Columns: {meta_df.columns.tolist()}"
        )

    META_COL = numeric_cols[0]

if len(meta_df) != N:
    raise ValueError(
        f"MetaStack row count {len(meta_df)} does not match "
        f"official test row count {N}."
    )

# Align the CSV to official test IDs.
# This prevents accidental use of a different CSV row order.
meta_aligned = meta_df.set_index("id").reindex(IDS)

if meta_aligned[META_COL].isna().any():
    raise ValueError(
        "MetaStack predictions contain missing official test IDs "
        "or null prediction values."
    )

META = meta_aligned[META_COL].to_numpy(dtype=np.float64)

# The source notebook assumes this .npy vector follows the
# official test.csv row order. It contains no IDs of its own.
XGB = np.asarray(
    np.load(XGB_PATH, allow_pickle=False),
    dtype=np.float64
).reshape(-1)

if len(META) != N:
    raise ValueError(
        f"MetaStack length mismatch: {len(META)} vs {N}."
    )

if len(XGB) != N:
    raise ValueError(
        f"XGBoost length mismatch: {len(XGB)} vs {N}."
    )

if not np.isfinite(META).all():
    raise ValueError("MetaStack has NaN or Inf values.")

if not np.isfinite(XGB).all():
    raise ValueError("XGBoost has NaN or Inf values.")

print("MetaStack prediction column:", META_COL)
print("MetaStack rows:", f"{len(META):,}")
print("XGBoost rows:", f"{len(XGB):,}")
print("MetaStack SHA-256:", sha256_file(META_PATH))
print("XGBoost SHA-256:", sha256_file(XGB_PATH))


# ------------------------------------------------------------
# 7. Reconstruct original u07_META105_xgb05 rank blend
# ------------------------------------------------------------

print("\n[5/6] Reconstructing original rank blend...")

def normalized_rank(values):
    values = np.asarray(values, dtype=np.float64).reshape(-1)

    if not np.isfinite(values).all():
        raise ValueError("Cannot rank non-finite predictions.")

    return rankdata(values) / len(values)


R_META = normalized_rank(META)
R_XGB = normalized_rank(XGB)

# Original recipe from Areeb's source notebook:
# u07_META105_xgb05
W_META = 1.05
W_XGB = -0.05

WEIGHT_SUM = W_META + W_XGB

if not np.isclose(WEIGHT_SUM, 1.0):
    raise ValueError(
        f"Unexpected original weight sum: {WEIGHT_SUM}"
    )

# Same weighted-rank blend formula as the source notebook.
R_BASELINE = (
    W_META * R_META
    + W_XGB * R_XGB
) / WEIGHT_SUM

if len(R_BASELINE) != N:
    raise ValueError("Baseline output length mismatch.")

if not np.isfinite(R_BASELINE).all():
    raise ValueError("Baseline output contains NaN or Inf.")

print("Candidate:", BASELINE_NAME)
print("Formula: (1.05 * rank(MetaStack) - 0.05 * rank(XGBoost)) / 1.00")
print("Output min:", float(R_BASELINE.min()))
print("Output max:", float(R_BASELINE.max()))


# ------------------------------------------------------------
# 8. Save immutable named baseline + submission.csv
# ------------------------------------------------------------

print("\n[6/6] Saving baseline and audit artifacts...")

baseline_path = OUT_DIR / f"{BASELINE_NAME}.csv"
submission_path = WORK_ROOT / "submission.csv"

baseline_df = pd.DataFrame({
    "id": IDS,
    "satisfaction": R_BASELINE
})

baseline_df.to_csv(baseline_path, index=False)

# Kaggle submission interface expects /kaggle/working/submission.csv.
shutil.copyfile(baseline_path, submission_path)

# Verify both files after writing.
check_baseline = pd.read_csv(baseline_path)
check_submission = pd.read_csv(submission_path)

for label, df in [
    ("named baseline", check_baseline),
    ("submission.csv", check_submission)
]:
    if df.columns.tolist() != ["id", "satisfaction"]:
        raise ValueError(
            f"{label}: unexpected columns {df.columns.tolist()}"
        )

    if len(df) != N:
        raise ValueError(
            f"{label}: incorrect row count {len(df)}"
        )

    if not np.array_equal(df["id"].to_numpy(), IDS):
        raise ValueError(
            f"{label}: ID order differs from official test.csv"
        )

    if not np.isfinite(
        df["satisfaction"].to_numpy(dtype=np.float64)
    ).all():
        raise ValueError(
            f"{label}: non-finite prediction values"
        )

if not np.array_equal(
    check_baseline["satisfaction"].to_numpy(),
    check_submission["satisfaction"].to_numpy()
):
    raise ValueError(
        "Named baseline and submission.csv predictions differ."
    )

baseline_sha = sha256_file(baseline_path)
submission_sha = sha256_file(submission_path)

if baseline_sha != submission_sha:
    raise ValueError(
        "Baseline and submission file hashes do not match."
    )


# Provenance manifest
manifest = {
    "experiment": "PAS_E40_Original_096217_Baseline",
    "candidate": BASELINE_NAME,
    "historical_public_lb_reference": HISTORICAL_PUBLIC_LB,
    "historical_score_recomputed": False,
    "formula": "1.05 * rank(MetaStack) - 0.05 * rank(XGBoost)",
    "rank_normalization": "scipy.stats.rankdata(prediction) / N",
    "rank_method": "scipy.stats.rankdata default average-rank behavior",
    "weights": {
        "MetaStack": W_META,
        "XGBoost": W_XGB
    },
    "n_test": int(N),
    "meta_prediction_column": META_COL,
    "sources": {
        "competition_test_csv": str(TEST_PATH),
        "competition_test_sha256": sha256_file(TEST_PATH),
        "meta_stack_submission_csv": str(META_PATH),
        "meta_stack_sha256": sha256_file(META_PATH),
        "xgboost_test_npy": str(XGB_PATH),
        "xgboost_sha256": sha256_file(XGB_PATH),
        "xgboost_row_order": (
            "Assumed to match official test.csv row order, "
            "as in the source notebook."
        )
    },
    "outputs": {
        "named_baseline": str(baseline_path),
        "named_baseline_sha256": baseline_sha,
        "submission_csv": str(submission_path),
        "submission_sha256": submission_sha
    },
    "uses_E37_or_E38_reconstructed_G": False,
    "uses_hblend_or_summit": False,
    "trains_models": False,
    "computes_oof_auc": False,
    "submits_to_kaggle": False
}

manifest_path = (
    OUT_DIR / "PAS_E40_BASELINE_MANIFEST.json"
)

manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

readme_path = OUT_DIR / "README.md"

readme_path.write_text(
    "# PAS E40 — Original 0.96217 Baseline\n\n"
    "Candidate: `u07_META105_xgb05`.\n\n"
    "Formula: `1.05 * normalized_rank(MetaStack) "
    "- 0.05 * normalized_rank(XGBoost)`.\n\n"
    "Required inputs: official S6E10 competition, Chris Deotte "
    "TFM predictions, and goodpjw2008 MetaStack Notebook output.\n\n"
    "This is a reconstruction of the source recipe, not a new "
    "leaderboard result. No OOF AUC is computed and no submission "
    "is performed automatically.\n",
    encoding="utf-8"
)

zip_path = (
    WORK_ROOT / "PAS_E40_Original_096217_Baseline_Assets.zip"
)

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=5
) as zf:
    zf.write(
        baseline_path,
        arcname=f"{BASELINE_NAME}.csv"
    )
    zf.write(
        manifest_path,
        arcname=manifest_path.name
    )
    zf.write(
        readme_path,
        arcname=readme_path.name
    )

print("\n" + "=" * 70)
print("BASELINE RECONSTRUCTION COMPLETE")
print("=" * 70)
print("Historical Public LB reference:", HISTORICAL_PUBLIC_LB)
print("Test rows:", f"{N:,}")
print("Named baseline:", baseline_path)
print("Kaggle submission file:", submission_path)
print("Baseline SHA-256:", baseline_sha)
print("Manifest:", manifest_path)
print("Result bundle:", zip_path)
print("Named baseline == submission.csv:", baseline_sha == submission_sha)
print("\nNo model training, OOF evaluation, or Kaggle submission was performed.")

display(baseline_df.head())
display(FileLink(str(zip_path)))
